***Installing dependencies***

In [ ]:
# Colab: نصب کتابخانه‌های مورد نیاز
!pip install -q transformers datasets accelerate peft evaluate

***report***

In [ ]:
import torch
import os
import time

def report_model_info(model, trainer, tag="Full Fine-tune", extra_info=None):
    print("="*80)
    print(f"📊 گزارش برای: {tag}")
    print("="*80)

    # معماری مدل
    cfg = model.config
    print("🔹 معماری مدل:")
    print(f"  - num_hidden_layers: {cfg.num_hidden_layers}")
    print(f"  - hidden_size: {cfg.hidden_size}")
    print(f"  - num_attention_heads: {cfg.num_attention_heads}")
    print()

    # هایپرپارامترها
    print("🔹 هایپرپارامترهای آموزش:")
    print(f"  - batch_size: {trainer.args.per_device_train_batch_size}")
    print(f"  - num_train_epochs: {trainer.args.num_train_epochs}")
    print(f"  - learning_rate: {trainer.args.learning_rate}")
    if extra_info:
        for k,v in extra_info.items():
            print(f"  - {k}: {v}")
    print()

    # تعداد پارامترهای آموزش‌پذیر
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print("🔹 پارامترها:")
    print(f"  - trainable parameters: {trainable:,}")
    print(f"  - total parameters: {total:,}")
    print(f"  - درصد پارامترهای آموزش‌پذیر: {100*trainable/total:.2f}%")
    print()

    # ارزیابی روی validation
    print("🔹 ارزیابی روی validation:")
    eval_metrics = trainer.evaluate()
    print(eval_metrics)
    print()

    # حجم فایل ذخیره‌شده
    ckpt_dir = trainer.args.output_dir
    if os.path.exists(ckpt_dir):
        size_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(ckpt_dir) for f in fn) / 1e6
        print(f"🔹 حجم پوشه‌ی مدل ذخیره‌شده: {size_mb:.2f} MB")
    else:
        print("⚠️ پوشه‌ی checkpoint پیدا نشد.")
    print("="*80, "\n")


***Loading libraries and initialization***

In [ ]:
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

# PEFT
from peft import LoraConfig, get_peft_model, TaskType, PromptTuningConfig, get_peft_model  # prompt tuning config هم داریم

# انتخاب مدل پایه
MODEL_NAME = "roberta-large"  # یا "FacebookAI/roberta-large"
NUM_LABELS = 3  # MNLI: entailment / neutral / contradiction

# متریک
accuracy = evaluate.load("accuracy")


***Dataset loading (MultiNLI) and 10% sampling***

In [ ]:
# بارگذاری MultiNLI از Hugging Face datasets
raw_datasets = load_dataset("multi_nli")

# برای صرفه‌جویی، فقط 10% از train را انتخاب می‌کنیم
train10pct = raw_datasets["train"].train_test_split(test_size=0.90, shuffle=True, seed=42)["train"]
# validation و test (dev_matched) را نگه‌داریم
validation = raw_datasets["validation_matched"]

print("train size (10%):", len(train10pct))
print("validation size:", len(validation))
print("\n📊 ساختار dataset:")
print(raw_datasets)

print("\n🔍 کلیدهای موجود:")
print(raw_datasets.keys())

# نمایش اطلاعات هر split
for split_name in raw_datasets.keys():
    print(f"\n📈 اطلاعات split '{split_name}':")
    print(f"   تعداد نمونه‌ها: {len(raw_datasets[split_name])}")
    print(f"   ویژگی‌ها: {raw_datasets[split_name].features}")

# نمایش 5 نمونه از داده آموزش (train)
print("\n" + "="*50)
print("🎯 5 نمونه از داده‌های آموزش (train):")
print("="*50)

train_df = raw_datasets['train'].to_pandas()
print(train_df.head(5))

# نمایش 5 نمونه از داده validation
print("\n" + "="*50)
print("🔍 5 نمونه از داده‌های validation (validation_matched):")
print("="*50)

validation_df = raw_datasets['validation_matched'].to_pandas()
print(validation_df.head(5))

# نمایش یک نمونه به صورت کامل با جزئیات
print("\n" + "="*50)
print("🔎 یک نمونه کامل با جزئیات:")
print("="*50)

sample = raw_datasets['train'][0]
for key, value in sample.items():
    print(f"🔑 {key}:")
    print(f"   {value}")
    print("-" * 30)

***Tokenizer and preprocessing***

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# 1️⃣ تعریف مدل / توکنایزر
MODEL_NAME = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2️⃣ تابع پردازش
def preprocess_function(examples):
    # premise = sentence1, hypothesis = sentence2
    return tokenizer(examples["premise"], examples["hypothesis"], truncation=True)

# 3️⃣ توکنایز کردن دیتاست
# ستون 'label' را نگه می‌داریم
encoded_train = train10pct.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in train10pct.column_names if c not in ["label"]]  # فقط ستون label نگه داشته میشه
)
encoded_val = validation.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in validation.column_names if c not in ["label"]]
)

# 4️⃣ rename ستون label به labels (مورد نیاز Trainer)
encoded_train = encoded_train.rename_column("label", "labels")
encoded_val   = encoded_val.rename_column("label", "labels")

# 5️⃣ فرمت PyTorch
encoded_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
encoded_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 6️⃣ data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



***compute_metrics function for Trainer***

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


***Scenario A — Full fine-tune (all parameters)***

***Preparing the model for classification (update all parameters)***

In [ ]:
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

***TrainingArguments and Trainer (Full)***

In [ ]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer

# 1️⃣ مدل برای classification
model_full = RobertaForSequenceClassification.from_pretrained(
    "roberta-large",
    num_labels=3
)

# 2️⃣ اطمینان از وجود ستون labels در dataset
# اگر ستون هنوز label هست، rename کن
if "label" in encoded_train.column_names:
    encoded_train = encoded_train.rename_column("label", "labels")
if "label" in encoded_val.column_names:
    encoded_val = encoded_val.rename_column("label", "labels")

# فرمت PyTorch
encoded_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
encoded_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 3️⃣ تعداد استپ‌ها در هر epoch
steps_per_epoch = len(encoded_train) // 4   # batch_size=4

# 4️⃣ آرگومان‌های آموزش
training_args_full = TrainingArguments(
    output_dir="./results/full_finetune",
    do_eval=True,                   # فعال کردن ارزیابی
    eval_steps=steps_per_epoch,      # معادل ارزیابی در پایان هر epoch
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=True,
    logging_steps=50,
    push_to_hub=False,
)

# 5️⃣ متریک accuracy
import evaluate
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6️⃣ تعریف Trainer
trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 7️⃣ آموزش
trainer_full.train()

# 8️⃣ گزارش
report_model_info(model_full, trainer_full, tag="Full Fine-tune")



***Scenario B — LoRA (PEFT) — Low-volume parameters\***

***Loading the LoRA model and configuring it***

In [ ]:
model_lora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# پیکربندی LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # چون داریم classification انجام میدیم
    inference_mode=False,
    r=8,           # rank کم (می‌تونی تغییر بدی)
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    # target_modules می‌تونه بسته به نام لایه‌های مدل تغییر کنه؛
    # برای RoBERTa معمولاً projection های attention را هدف قرار می‌دهیم.
    target_modules=["query", "value"]
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()  # نمایش درصد/تعداد پارامترهای آموزشی


***TrainingArguments and Trainer for LoRA***

In [ ]:
from transformers import TrainingArguments, Trainer

# تعداد استپ‌ها در هر epoch
steps_per_epoch = len(encoded_train) // 8  # batch_size=8

training_args_lora = TrainingArguments(
    output_dir="./results/lora_finetune",
    do_eval=True,                   # فعال کردن ارزیابی
    eval_steps=steps_per_epoch,      # معادل ارزیابی در پایان هر epoch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=3e-4,  # معمولاً lr برای LoRA بالاتر است
    weight_decay=0.0,
    save_total_limit=1,
    fp16=True,
    logging_steps=50,
    push_to_hub=False,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_lora.train()
report_model_info(model_lora, trainer_lora, tag="LoRA", extra_info={"lora_r": 8})


 ***Scenario C—Prompt Tuning (Soft-Prompt / Tuning-P) with PEFT***

***Configure PromptTuning and model wrap***

In [ ]:
model_prompt = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=50,  # تعداد توکن‌هایی که به‌عنوان soft prompt می‌سازیم
    token_dim=model_prompt.config.hidden_size
)

# PEFT API: برخی نسخه‌ها PromptTuningConfig را متفاوت می‌پذیرند؛ اگر خطا دیدی مستندات PEFT را برای نسخه‌ات بررسی کن.
model_prompt = get_peft_model(model_prompt, prompt_config)
model_prompt.print_trainable_parameters()


***TrainingArguments and Trainer for PromptTuning***

In [ ]:
from transformers import TrainingArguments, Trainer

# تعداد استپ‌ها در هر epoch
steps_per_epoch = len(encoded_train) // 16  # batch_size=16

training_args_prompt = TrainingArguments(
    output_dir="./results/prompt_tuning",
    do_eval=True,                   # فعال کردن ارزیابی
    eval_steps=steps_per_epoch,      # معادل ارزیابی در پایان هر epoch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,  # prompt tuning ممکنه نیاز به epochs بیشتری داشته باشه
    learning_rate=5e-4,
    weight_decay=0.0,
    save_total_limit=1,
    fp16=True,
    logging_steps=50,
    push_to_hub=False,
)

trainer_prompt = Trainer(
    model=model_prompt,
    args=training_args_prompt,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_prompt.train()

report_model_info(model_prompt, trainer_prompt, tag="Prompt Tuning", extra_info={"num_virtual_tokens": 50})
